In [7]:
# Install if needed (comment if already installed)
!pip install google-generativeai nest_asyncio

In [8]:
import os
import nest_asyncio
nest_asyncio.apply()
!pip install google-genai nest_asyncio -q
#import google.generativeai as genai
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("googleKey")

In [9]:
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

In [10]:
# 🔹 Memory store
memory = []

# 🔹 Gemini wrapper (NEW SDK)
def gemini_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text


# 🔹 Agent class with memory awareness
class GeminiAgent:
    def __init__(self, name):
        self.name = name

    def generate_reply(self, message):
        # Include memory in prompt
        context = "\n".join(memory[-5:])  # last 5 interactions
        full_prompt = f"""
        Previous context:
        {context}

        Current task:
        {message}
        """

        print(f"\n[{self.name}] thinking...")
        response = gemini_llm(full_prompt)

        print(f"\n[{self.name}]:\n{response}")

        # Save to memory
        memory.append(f"{self.name}: {response}")

        return response


# 🔹 Create agents
planner = GeminiAgent("Planner")
assistant = GeminiAgent("Assistant")


# 🔹 Run workflow with memory
def run_agents(task):
    print("\n🟢 USER TASK:\n", task)

    memory.append(f"User: {task}")

    # Step 1: Planning
    plan = planner.generate_reply(f"Break this task into steps:\n{task}")

    # Step 2: Execution
    result = assistant.generate_reply(f"Execute this plan:\n{plan}")

    print("\n✅ FINAL OUTPUT:\n", result)

In [11]:
# 🔹 Run
run_agents("Write a Python program to compute Fibonacci numbers up to 10")


🟢 USER TASK:
 Write a Python program to compute Fibonacci numbers up to 10

[Planner] thinking...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
run_agents("My name is Narasimhan")
run_agents("What is my name?")

In [ ]:
print("\n🔹 MEMORY STATE:\n", memory)